In [ ]:
CREATE OR REPLACE TABLE memory.authors AS
SELECT DISTINCT id AS author_id,
        unnest(display_name_alternatives) AS author_name,
    FROM authors.authors;

CREATE OR REPLACE TABLE memory.institutions AS
SELECT DISTINCT id AS author_id,
                unnest(last_known_institutions).id AS institution_id,
                unnest(last_known_institutions).display_name AS institution_name,
                unnest(last_known_institutions).country_code AS country_code,
    FROM authors.authors;

CREATE OR REPLACE TABLE memory.topics AS
  SELECT DISTINCT * FROM
  (
  SELECT id AS author_id, 
        display_name AS author_name,
        works_count,
        cited_by_count,
        summary_stats.h_index,
        replace(unnest(topics).field.id, 'https://openalex.org/fields/', '') AS field_id,
        unnest(topics).field.display_name AS field_name
  FROM authors.authors
    WHERE len(topics) != 0
  )
  WHERE field_id IN (14, 20);

CREATE OR REPLACE TABLE econ.authors AS
SELECT t.*,
        a.author_name,
        i.institution_id,
        i.institution_name,
        i.country_code
  FROM memory.topics t
  LEFT JOIN memory.authors a
  ON t.author_id = a.author_id
  LEFT JOIN memory.institutions i
  ON t.author_id = i.author_id

In [ ]:
SELECT count(sub.work_id) AS cited_by_count_endogenous,
        sum(sub.cited_by_count) AS cited_by_count_total,
        sub.author_id,
        sub.author_name,
        sub.publication_year
  FROM
  (SELECT DISTINCT w.work_id,
        w.cited_by_count,
        a.author_id,
        a.author_name,
        w.publication_year
  FROM works w
  LEFT JOIN (SELECT work_id,
            unnest(referenced_works) AS cited_id
            FROM cited
          ) c
  ON w.work_id = c.cited_id
  LEFT JOIN authorships a
  ON a.work_id = w.work_id
  WHERE a.author_id NOT NULL) sub
  GROUP BY ALL
ORDER BY cited_by_count_endogenous DESC

In [ ]:
SELECT count(sub.work_id) AS cited_by_count_endogenous,
        sum(sub.cited_by_count) AS cited_by_count_total,
        sub.author_id,
        sub.author_name,
        sub.publication_year
  FROM
  (SELECT DISTINCT w.work_id,
        w.cited_by_count,
        a.author_id,
        a.author_name,
        w.publication_year
  FROM works w
  LEFT JOIN (SELECT work_id,
            unnest(referenced_works) AS cited_id
            FROM cited
          ) c
  ON w.work_id = c.cited_id
  LEFT JOIN authorships a
  ON a.work_id = w.work_id
  WHERE a.author_id NOT NULL) sub
  GROUP BY ALL
ORDER BY cited_by_count_endogenous DESC

In [ ]:
CREATE OR REPLACE TABLE memory.citations_per_work AS
SELECT c.cited_id,
        count(c.citer_id) AS cited_by_count_endogenous,
        w.cited_by_count AS cited_by_count_total,
        w.publication_year
  FROM
  (SELECT work_id AS citer_id,
          unnest(referenced_works) AS cited_id
    FROM cited
    ) c
  LEFT JOIN works w
  ON c.cited_id = w.work_id
  WHERE w.work_id NOT NULL
  GROUP BY ALL
  ORDER BY publication_year, cited_by_count_endogenous DESC, cited_by_count_total DESC;

CREATE OR REPLACE TABLE memory.citations_per_work_ranked AS
  SELECT cited_id,
        publication_year,
        cited_by_count_total,
        cited_by_count_endogenous,
        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous
  FROM memory.citations_per_work
  WINDOW w AS (PARTITION BY publication_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
  ORDER BY publication_year DESC, percent_rank_total DESC;

CREATE OR REPLACE TABLE memory.citations AS
  SELECT author_id,
        author_name,
        sum(cited_by_count_total) AS citations_total,
        sum(cited_by_count_endogenous) AS citations_endogenous
  FROM memory.citations_per_work_ranked m
  LEFT JOIN authorships a
  ON m.cited_id = a.work_id
  WHERE author_id NOT NULL
  GROUP BY author_id, author_name
  ;

CREATE OR REPLACE TABLE memory.hca_endogenous AS
  SELECT author_id,
        author_name,
        count(cited_id) AS hca_endogenous
  FROM memory.citations_per_work_ranked m
  LEFT JOIN authorships a
  ON m.cited_id = a.work_id
  WHERE a.work_id NOT NULL 
          AND percent_rank_endogenous >= 0.99
  GROUP BY ALL
    ORDER BY hca_endogenous DESC;

CREATE OR REPLACE TABLE memory.hca_total AS
  SELECT author_id,
        author_name,
        count(cited_id) AS hca_total,
  FROM memory.citations_per_work_ranked m
  LEFT JOIN authorships a
  ON m.cited_id = a.work_id
  WHERE a.work_id NOT NULL 
          AND percent_rank_total >= 0.99
  GROUP BY ALL
  ORDER BY hca_total DESC;

CREATE OR REPLACE TABLE memory.citations_endogenous_all AS
  SELECT author_id,
        author_name,
        citations_total,
        citations_endogenous,
        hca_total,
        hca_endogenous
  FROM memory.citations
  LEFT JOIN
    (SELECT t.*,
            e.hca_endogenous
      FROM memory.hca_total t
      LEFT JOIN memory.hca_endogenous e
      USING (author_id)) sub
    USING (author_id, author_name)
  ORDER BY citations_total DESC;
CREATE OR REPLACE TABLE memory.author_works_counts AS
  SELECT au.author_id,
        au.author_name,
        a.orcid,
        display_name_alternatives,
        count(work_id) AS works_count_endogenous,
        works_count,
        cited_by_count,
        "2yr_mean_citedness",
        h_index      
    FROM econ.authorships au
    LEFT JOIN econ.authors a
      ON a.author_id = au.author_id
      --WHERE a.author_id = 'https://openalex.org/A5012301204'
    GROUP BY ALL
    ORDER BY works_count DESC;

CREATE OR REPLACE TABLE econ.citation_summary AS
SELECT DISTINCT c.author_id,
        c.author_name,
        a.author_name,
        a.works_count_endogenous,
        s.citations_total AS citations_total_,
        c.citations_endogenous,
        hca_total,
        hca_endogenous,
        a.orcid,
        a.display_name_alternatives,
        a.works_count AS works_count_total,
        a.cited_by_count,
        a."2yr_mean_citedness",
        a.h_index
FROM memory.citations c
  LEFT JOIN econ.citations_endogenous_all s
  ON c.author_id = s.author_id
  LEFT JOIN memory.author_works_counts a
  ON c.author_id = a.author_id
  ORDER BY c.citations_total DESC, h_index DESC;

SELECT DISTINCT *
  FROM econ.authors
  ORDER BY works_count DESC;


In [ ]:
WITH total_works_cte AS
  (SELECT count(DISTINCT work_id) AS total_works FROM econ.works)
SELECT DISTINCT p.pageRank, w.source_id, pageRank*(SELECT * FROM total_works_cte)/count(work_id)/100 AS influence
    FROM econ.pagerank_sources p
    LEFT JOIN econ.works w
    ON p.citer = w.source_id
    GROUP BY p.pageRank, w.source_id;

WITH 
  total_works_cte AS
    (SELECT count(DISTINCT work_id) AS total_works FROM econ.works),
  work_institution_cte AS
    (SELECT DISTINCT work_id, institution_id 
      FROM econ.works w
      LEFT JOIN econ.authorships a
      USING (work_id))
SELECT DISTINCT institution_id, pageRank*(SELECT * FROM total_works_cte)/count(work_id)/100 AS influence
    FROM econ.pagerank_institutions p
    LEFT JOIN work_institution_cte w
    ON p.citer = w.institution_id
    GROUP BY pageRank, institution_id


-- WITH 
--   work_institution_cte AS
--     (SELECT DISTINCT work_id, institution_id 
--       FROM econ.works w
--       LEFT JOIN econ.authorships a
--       USING (work_id)),
--   pagerank_institution_cte AS
--       (SELECT pagerank, institution_id, count(work_id) AS works_count
--         FROM econ.pagerank_institutions p
--         LEFT JOIN
--         (SELECT work_id, institution_id 
--         FROM work_institution_cte) sub
--         ON sub.institution_id = p.citer
--         GROUP BY ALL)
--   SELECT institution_id, pageRank, pagerank*sum(works_count)/works_count AS influence_institutions
--     FROM pagerank_institution_cte
--     GROUP BY institution_id, pagerank, works_count

In [ ]:
-- ENDOGENOUS H-Index
SELECT cited_author, 
        count(*) AS endogenous_h_index
  FROM
    (SELECT count(citer_id) AS citer_count,
        cited_id,
        author_id AS cited_author,
        rank() OVER (PARTITION BY author_id ORDER BY count(citer_id)) ranking
    FROM
      (SELECT work_id AS citer_id,
          unnest(referenced_works) AS cited_id
      FROM econ.cited) cc
      LEFT JOIN econ.authorships a
      ON cc.cited_id = a.work_id
      WHERE cc.cited_id NOT NULL and a.author_id NOT NULL
    GROUP BY author_id, cited_id)
  WHERE citer_count >= ranking
  GROUP BY cited_author
  ORDER BY endogenous_h_index DESC;

SELECT DISTINCT a1.work_id,
        a1.author_id AS author_1,
        a2.author_id AS author_2
  FROM econ.authorships a1
  LEFT JOIN econ.authorships a2
  USING (work_id)
  WHERE author_1 != author_2
  ORDER BY a1.work_id, a1.author_id, a2.author_id

In [ ]:
WITH 
  citer_cited_etl AS	
    (SELECT work_id AS citer_id, unnest(referenced_works) AS cited_id
      FROM econ.cited),
  
  endogenous_citer_cited_etl AS
    (SELECT citer_id, cited_id
      FROM citer_cited_etl
      LEFT JOIN econ.works cited_works
      ON cited_works.work_id = cited_id
      WHERE cited_id NOT NULL),

  endogenous_citer_cited_author_etl AS
    (SELECT DISTINCT citer_id,
            cited_id,
            citer_a.author_id AS citer_author,
            cited_a.author_id AS cited_author
      FROM endogenous_citer_cited_etl ecc
      LEFT JOIN econ.authorships citer_a
      ON ecc.citer_id = citer_a.work_id
      LEFT JOIN econ.authorships cited_a
      ON ecc.cited_id = cited_a.work_id
      WHERE citer_a.author_id NOT NULL AND cited_a.author_id NOT NULL),
  
  -- endogenous_citer_cited_author_etl AS
  --   (SELECT DISTINCT citer_id,
  --           citer_a.author_id AS citer_author,
  --           cited_id,
  --           cited_a.author_id AS cited_author
  --     FROM endogenous_citer_cited_py_etl eccp
  --     LEFT JOIN econ.authorships citer_a
  --     ON eccp.citer_id = citer_a.work_id
  --     LEFT JOIN econ.authorships cited_a
  --     ON eccp.cited_id = cited_a.work_id
  --     WHERE citer_a.author_id NOT NULL AND cited_a.author_id NOT NULL),
  
  endogenous_citer_cited_self_etl AS
    (SELECT DISTINCT cited_id,
              list_has_any(list(citer_author), list(cited_author)) AS self_cited
      FROM endogenous_citer_cited_author_etl
      GROUP BY citer_author, cited_id, cited_author),
  
  endogenous_citations_author_etl AS
    (SELECT count(a.citer_id) AS citations,
            sum(CASE WHEN self_cited=true THEN 1 ELSE 0 END) AS citations_self, 
            sum(CASE WHEN self_cited=true THEN 1 ELSE 0 END)/count(a.citer_id) AS citations_self_ratio, 
            cited_author 
      FROM endogenous_citer_cited_author_etl a
      LEFT JOIN endogenous_citer_cited_self_etl s
        ON a.cited_id = s.cited_id
      GROUP BY cited_author)

SELECT * 
  FROM endogenous_citations_author_etl

  -- SELECT cited_author, 
  --           cited_id, 
  --           count(citer_id) AS citations_count,
  --           row_number() OVER (PARTITION by cited_author, cited_id ORDER BY count(citer_id) DESC) AS ranking

  -- SELECT *
  --   FROM
  --     (SELECT count(citer_id),
  --             cited_author,
  --             cited_id,
  --             row_number() OVER (PARTITION BY cited_author, cited_id ORDER BY count(citer_id) DESC) AS ranking
  --       FROM
  --         (SELECT work_id, 
  --                   author_id AS cited_author
  --           FROM econ.works
  --           LEFT JOIN econ.authorships
  --           USING (work_id)) sub
  --       RIGHT OUTER JOIN citer_cited_etl cc
  --       ON sub.work_id = cc.cited_id
  --       WHERE citer_id NOT NULL AND cited_id NOT NULL 
  --       GROUP BY cited_author, cited_id)
  
        
      -- GROUP BY cited_author, cited_id
  
  -- SELECT cited_author, 
  --         count(*) AS h_index,
  --         count(cited_id) AS cited_works_count,
  --         sum(citations_count) AS citations_total,
  --         sum(citations_count)/(count(*)*count(*)) AS c_on_h_squared
  --   FROM
  --     (SELECT cited_author, 
  --           cited_id, 
  --           count(citer_id) AS citations_count,
  --           rank() OVER (PARTITION by cited_author, cited_id ORDER BY count(citer_id) DESC) AS ranking
  --     FROM endogenous_citer_cited_author_etl
  --     GROUP BY cited_author, cited_id) sub
  --   WHERE ranking <= citations_count
  --   GROUP BY cited_author

-- SELECT citations_count,
--         cited_author,
--   FROM endogenous_citer_cited_author_etl
--     LEFT JOIN endogenous_citations_author_etl
--     USING (cited_author)

-- select scientistid, count(*)
-- from (SELECT p.scientistid, p.id, COUNT(c.id) AS citations_count,
--              rank() over (partition by p.scientistid, p.id order by count(c.id) desc) as ranking
--       FROM PAPERS p LEFT OUTER JOIN
--            CITATIONS c
--            ON p.id = c.paper_id
--       GROUP BY p.scientist, p.id
--      ) t)
-- where ranking <= citations_count
-- group by author_id;

In [ ]:
WITH 
  works_data_etl AS
    (SELECT work_id, publication_year,  
      FROM econ.works),
  citer_cited_etl AS	
    (SELECT work_id AS citer_id, unnest(referenced_works) AS cited_id
      FROM econ.cited),
  endogenous_citer_cited_etl AS
    (SELECT citer_id, cited_id
      FROM citer_cited_etl
      LEFT JOIN works_data_etl cited_works
      ON cited_works.work_id = cited_id
      WHERE cited_id NOT NULL),
  endogenous_citer_cited_py_etl AS
    (SELECT citer_id, 
            citer_works.publication_year AS citer_publication_year,
            cited_id,
            cited_works.publication_year AS cited_publication_year
      FROM endogenous_citer_cited_etl
      LEFT JOIN works_data_etl citer_works
      ON citer_id = citer_works.work_id
      LEFT JOIN works_data_etl cited_works
      ON cited_id = cited_works.work_id),
  endogenous_citer_cited_py_author_etl AS
    (SELECT DISTINCT citer_id,
            citer_publication_year,
            citer_a.author_id AS citer_author,
            cited_id,
            cited_publication_year,
            cited_a.author_id AS cited_author
      FROM endogenous_citer_cited_py_etl eccp
      LEFT JOIN econ.authorships citer_a
      ON eccp.citer_id = citer_a.work_id
      LEFT JOIN econ.authorships cited_a
      ON eccp.cited_id = cited_a.work_id
      WHERE citer_a.author_id NOT NULL AND cited_a.author_id NOT NULL),
  endogenous_citer_cited_author_etl AS
    (SELECT DISTINCT citer_id,
            citer_a.author_id AS citer_author,
            cited_id,
            cited_a.author_id AS cited_author
      FROM endogenous_citer_cited_py_etl eccp
      LEFT JOIN econ.authorships citer_a
      ON eccp.citer_id = citer_a.work_id
      LEFT JOIN econ.authorships cited_a
      ON eccp.cited_id = cited_a.work_id
      WHERE citer_a.author_id NOT NULL AND cited_a.author_id NOT NULL),
  endogenous_citer_cited_self_etl AS
    (SELECT DISTINCT cited_id,
              list_has_any(list(citer_author), list(cited_author)) AS self_cited
      FROM endogenous_citer_cited_py_author_etl
      GROUP BY citer_author, cited_id, cited_author),
  endogenous_citations_author_etl AS
    (SELECT count(a.citer_id) AS citations,
            sum(CASE WHEN self_cited=true THEN 1 ELSE 0 END) AS citations_self, 
            sum(CASE WHEN self_cited=true THEN 1 ELSE 0 END)/count(a.citer_id) AS citations_self_cited_ratio, 
            cited_author 
      FROM endogenous_citer_cited_py_author_etl a
      LEFT JOIN endogenous_citer_cited_self_etl s
        ON a.cited_id = s.cited_id
      GROUP BY cited_author)
  -- SELECT * FROM endogenous_citations_author_etl

In [ ]:
WITH
  author_works_list_ETL AS
    (SELECT author_id AS target_id,
            list(work_id) AS works,
            count(DISTINCT work_id) AS works_count
      FROM (SELECT DISTINCT author_id, work_id
            FROM econ.authorships)
      GROUP BY author_id),
  author_works_ETL AS
        (SELECT target_id,
              unnest(works) AS work_id
        FROM author_works_list_ETL),
  coauthors_ETL AS
    (SELECT DISTINCT target_id,
            work_id,
            author_id
      FROM author_works_ETL
        LEFT JOIN econ.authorships
        USING (work_id)
        WHERE target_id != author_id),
  coauthor_counts_ETL AS
    (SELECT target_id,
            count(author_id) AS coauthor_instances,
      FROM coauthors_ETL
      GROUP BY target_id, author_id),
  ranking_ETL AS
    (SELECT target_id,
              coauthor_instances,
              row_number() OVER (PARTITION BY target_id ORDER BY coauthor_instances DESC) AS ranking
      FROM coauthor_counts_ETL
      GROUP BY target_id, coauthor_instances
      ORDER BY target_id, ranking)

  SELECT target_id,
          sum(coauthor_instances) AS coauthors_unique,
          1.0-2.0*sum((coauthor_instances*(ranking-1) + coauthor_instances/2))/count(*)/sum(coauthor_instances) AS gini_coauthors
    FROM ranking_ETL
    GROUP BY target_id
    ORDER BY gini_coauthors DESC

In [ ]:
WITH
  citer_cited_ETL AS
    (SELECT work_id AS citer_id,
          unnest(referenced_works) AS cited_id
    FROM econ.cited),
  authors_works_ETL AS
    (SELECT DISTINCT work_id, 
              author_id
    FROM econ.authorships
    WHERE author_id NOT NULL),
  citer_cited_authors_works_ETL AS
    (SELECT citer_id,
            aw1.author_id AS citer_author,
            cited_id,
            aw2.author_id AS cited_author
      FROM citer_cited_ETL cc
      LEFT JOIN authors_works_ETL aw1
      ON aw1.work_id = cc.citer_id
      LEFT JOIN authors_works_ETL aw2
      ON aw2.work_id = cc.cited_id
      WHERE citer_author NOT NULL AND cited_author NOT NULL
      ORDER BY cited_author, citer_author),
  citer_ranking_ETL AS
    (SELECT cited_author,
              citer_author,
              count(citer_id) AS citer_count,
              row_number() OVER (PARTITION BY citer_author ORDER BY count(citer_id) DESC) AS ranking
      FROM citer_cited_authors_works_ETL
      GROUP BY cited_author, citer_author),

  
  -- SELECT cited_author,
  --         sum(citer_count) AS cited_total,
  --         1.0-2.0*sum((citer_count*(ranking-1) + citer_count/2))/count(*)/sum(citer_count) AS gini_cited
  --   FROM
  --   (SELECT cited_author,
  --           citer_author,
  --           count(citer_id) AS citer_count,
  --           row_number() OVER (PARTITION BY cited_author ORDER BY count(citer_id) DESC) AS ranking
  --     FROM citer_cited_authors_works_ETL
  --     GROUP BY cited_author, citer_author)
  --   GROUP BY cited_author
  --   HAVING sum(citer_count) > 0
  -- ORDER BY gini_cited DESC, cited_author

  -- SELECT citer_author,
  --         sum(cited_count) AS citer_total,
  --         1.0-2.0*sum((cited_count*(ranking-1) + cited_count/2))/count(*)/sum(cited_count) AS gini_citer
  --   FROM
  --   (SELECT citer_author,
  --           cited_author,
  --           count(cited_id) AS cited_count,
  --           row_number() OVER (PARTITION BY citer_author ORDER BY count(cited_id) DESC) AS ranking
  --     FROM citer_cited_authors_works_ETL
  --     GROUP BY citer_author, cited_author)
  --   GROUP BY citer_author
  --   HAVING sum(cited_count) > 0
  -- ORDER BY gini_citer DESC, citer_author